# Analyse: Kamerastabilität


In [1]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

DATA_DIR = Path('../../data')
COMBINED_CSV = DATA_DIR / '03_datasets/influencer_balanced/01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50.csv'
OUTPUT_CSV = DATA_DIR / '04_analysis_results' / 'visual_features' / '14_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_camera_stability.csv'

# Kombinierten MP4-Ordner bevorzugen; sonst quellenspezifische Ordner nutzen
VIDEO_ROOT = DATA_DIR / '02_media/stratified_sample/videos'
VIDEO_ROOTS_FALLBACK = {
    'ai': DATA_DIR / '02_media/ai_videos_2025/videos',
    'real': DATA_DIR / '02_media/real_videos_2025/videos',
}

SOURCE_FILTER = None
MAX_FRAME_PAIRS_PER_VIDEO = 80
TARGET_ANALYSIS_FPS = 3.0
MAX_FRAME_WIDTH = 480
MIN_TRACKED_POINTS = 8
QUALITY_LEVEL = 0.01
MIN_DISTANCE = 8

df = pd.read_csv(COMBINED_CSV)
if SOURCE_FILTER:
    df = df[df['influencer_type'].isin(SOURCE_FILTER)].copy()

video_id_col = 'video_id' if 'video_id' in df.columns else 'id'
df[video_id_col] = df[video_id_col].astype(str)
print(f'Loaded {len(df)} rows from {COMBINED_CSV.name}')


Loaded 500 rows from 01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50.csv


In [2]:
METRIC_COLUMNS = [
    'optical_flow_magnitude_mean',
    'optical_flow_magnitude_std',
    'stability_index',
    'processed_frame_pairs',
]

def resolve_video_path(video_id: str, source: str | None) -> Path | None:
    primary = VIDEO_ROOT / f'{video_id}.mp4'
    if primary.is_file():
        return primary

    fallback_root = VIDEO_ROOTS_FALLBACK.get(source)
    if fallback_root is not None:
        fallback = fallback_root / f'{video_id}.mp4'
        if fallback.is_file():
            return fallback

    return None

def has_video(video_id: str, source: str | None) -> bool:
    return resolve_video_path(video_id, source) is not None

df['has_video'] = [
    has_video(str(row[video_id_col]), row.get('influencer_type'))
    for _, row in df.iterrows()
]

missing_ids = df.loc[~df['has_video'], video_id_col].astype(str).tolist()
print(f"Videos with video file: {df['has_video'].sum()}")
print(f'Videos missing video file: {len(missing_ids)}')
if missing_ids:
    print('First missing video_ids:', missing_ids[:20])

df = df[df['has_video']].reset_index(drop=True)
print(f'Restricted to {len(df)} rows with available videos')


Videos with video file: 500
Videos missing video file: 0
Restricted to 500 rows with available videos


In [3]:
def prepare_gray_frame(frame: np.ndarray, max_frame_width: int = MAX_FRAME_WIDTH) -> np.ndarray:
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    height, width = gray.shape[:2]
    if width > max_frame_width:
        scale = max_frame_width / width
        gray = cv2.resize(gray, (int(width * scale), int(height * scale)), interpolation=cv2.INTER_AREA)
    return gray

def compute_global_motion(prev_gray: np.ndarray, curr_gray: np.ndarray) -> float:
    prev_points = cv2.goodFeaturesToTrack(
        prev_gray,
        maxCorners=300,
        qualityLevel=QUALITY_LEVEL,
        minDistance=MIN_DISTANCE,
        blockSize=7,
    )

    if prev_points is None or len(prev_points) < MIN_TRACKED_POINTS:
        return float('nan')

    curr_points, status, _ = cv2.calcOpticalFlowPyrLK(
        prev_gray,
        curr_gray,
        prev_points,
        None,
        winSize=(21, 21),
        maxLevel=3,
        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01),
    )

    if curr_points is None or status is None:
        return float('nan')

    valid = status.reshape(-1) == 1
    if valid.sum() < MIN_TRACKED_POINTS:
        return float('nan')

    displacements = curr_points[valid] - prev_points[valid]
    magnitudes = np.linalg.norm(displacements, axis=1)
    frame_diagonal = float(np.hypot(prev_gray.shape[1], prev_gray.shape[0]))
    if frame_diagonal <= 0:
        return float('nan')

    # Robuste globale Bewegungsmessung: Median der Punktverschiebung,
    # normiert über die Bilddiagonale für vergleichbare Auflösungen.
    return float(np.median(magnitudes) / frame_diagonal)

def measure_camera_stability(video_path: Path) -> dict:
    metrics = {col: np.nan for col in METRIC_COLUMNS}

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return metrics

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    if fps <= 0:
        fps = 30.0

    frame_step = max(1, int(round(fps / TARGET_ANALYSIS_FPS)))
    expected_pairs = frame_count // frame_step if frame_count > 0 else MAX_FRAME_PAIRS_PER_VIDEO
    target_pairs = max(1, min(MAX_FRAME_PAIRS_PER_VIDEO, expected_pairs))

    motions = []
    prev_gray = None
    frame_idx = 0

    while len(motions) < target_pairs:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % frame_step != 0:
            frame_idx += 1
            continue

        curr_gray = prepare_gray_frame(frame)
        if prev_gray is not None:
            motion = compute_global_motion(prev_gray, curr_gray)
            if pd.notna(motion):
                motions.append(motion)

        prev_gray = curr_gray
        frame_idx += 1

    cap.release()

    if motions:
        values = np.asarray(motions, dtype=float)
        mean_motion = float(values.mean())
        metrics['optical_flow_magnitude_mean'] = mean_motion
        metrics['optical_flow_magnitude_std'] = float(values.std())
        metrics['processed_frame_pairs'] = int(len(values))
        metrics['stability_index'] = float(1.0 / (1.0 + (mean_motion * 100.0)))

    return metrics


In [4]:
result_rows = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    video_id = str(row[video_id_col])
    source = row.get('influencer_type')
    video_path = resolve_video_path(video_id, source)

    metrics = {'video_id': video_id}
    for col in METRIC_COLUMNS:
        metrics[col] = np.nan

    if video_path is not None:
        metrics.update(measure_camera_stability(video_path))

    result_rows.append(metrics)

feature_df = pd.DataFrame(result_rows)
merged = df.merge(feature_df, on='video_id', how='left') if 'video_id' in df.columns else df.merge(feature_df, left_on=video_id_col, right_on='video_id', how='left')
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
merged.to_csv(OUTPUT_CSV, index=False)

summary_cols = [col for col in ['influencer_type'] + METRIC_COLUMNS if col in merged.columns]
print(merged[summary_cols].groupby('influencer_type').agg(['count', 'mean', 'std']).round(4))
print(f'Wrote {OUTPUT_CSV} with shape {merged.shape}')
merged.head()


100%|██████████| 500/500 [11:01<00:00,  1.32s/it]

                optical_flow_magnitude_mean                  \
                                      count    mean     std   
influencer_type                                               
ai                                      250  0.0057  0.0074   
real                                    250  0.0105  0.0089   

                optical_flow_magnitude_std                 stability_index  \
                                     count    mean     std           count   
influencer_type                                                              
ai                                     250  0.0084  0.0104             250   
real                                   250  0.0138  0.0108             250   

                                processed_frame_pairs                   
                   mean     std                 count    mean      std  
influencer_type                                                         
ai               0.7288  0.2191                   250  41.944  21.8479  
r

,video_id,video_thread_id,author_username,author_displayname,author_follower_count,author_like_count_total,author_video_count,author_avatar_url,video_caption,video_stickers,...,video_duration_seconds,influencer,influencer_clean,video_engagement_rate,eng_bin,has_video,optical_flow_magnitude_mean,optical_flow_magnitude_std,stability_index,processed_frame_pairs
0,7516243181650988334,7516243181650988334,ai.kalai,kalai.ai,9773,20,55,https://p16-sign-va.tiktokcdn.com/tos-maliva-a...,oop 🫢🫢 #fyp #trending #googleveo3 #ai #viral ...,🫢,...,6.958,ai.kalai,ai.kalai,0.011906,Low,True,0.003794,0.016261,0.724953,20
1,7515925552549678378,7515925552549678378,ai.kalai,kalai.ai,9773,20,55,https://p16-sign-va.tiktokcdn.com/tos-maliva-a...,"and if i’m fake, i ain’t notice cause my money...",NaN,...,9.500,ai.kalai,ai.kalai,0.013038,Low,True,0.003678,0.003861,0.731078,28
2,7521159314757684535,7521159314757684535,ai.kalai,kalai.ai,9773,20,55,https://p16-sign-va.tiktokcdn.com/tos-maliva-a...,uh oh!! #peanutbutter #groceryshopping #grocer...,teaching yall how i steal cause why this almon...,...,8.000,ai.kalai,ai.kalai,0.006939,Low,True,0.003573,0.002274,0.736742,23
3,7521486299098778894,7521486299098778894,ai.kalai,kalai.ai,9773,20,55,https://p16-sign-va.tiktokcdn.com/tos-maliva-a...,insanee #movies #movienight #popcorn #fastfood...,NaN,...,7.300,ai.kalai,ai.kalai,0.010583,Low,True,0.018908,0.034068,0.345928,20
4,7521490969468865847,7521490969468865847,ai.kalai,kalai.ai,9773,20,55,https://p16-sign-va.tiktokcdn.com/tos-maliva-a...,and he knows that 🎬 #acting #actor #actress #v...,NaN,...,7.967,ai.kalai,ai.kalai,0.009967,Low,True,0.003422,0.005398,0.745047,23
